# Advanced Retrieval: Hybrid Search with BM25

## Learning Objectives
- Understand limitations of pure semantic search
- Implement BM25 text search
- Combine BM25 + semantic search with fusion techniques
- Advanced chunking strategies
- Retrieval evaluation

## Prerequisites
- Completed Session 1 (RAG Fundamentals)
- Understanding of embeddings and vector search

In [ ]:
import os
import numpy as np
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv()
genai.configure(api_key=os.getenv('GEMINI_API_KEY'))

# Load sample documents
def load_sample_documents():
    """Load sample text files from data directory."""
    documents = []
    data_dir = "data/sample_texts"
    
    for filename in os.listdir(data_dir):
        if filename.endswith('.txt'):
            filepath = os.path.join(data_dir, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                text = f.read()
                documents.append({
                    'text': text,
                    'metadata': {'source': filename}
                })
    
    return documents

docs = load_sample_documents()
print("Session 2: Advanced Retrieval")
print(f"Loaded {len(docs)} documents")

## 1. Limitations of Pure Semantic Search

Semantic search is powerful but has blind spots:
- Misses exact keyword matches
- Struggles with technical terms
- Can't handle typos well
- May not capture domain-specific jargon

Let's see an example:

In [ ]:
# Example: Semantic search misses exact matches
import chromadb

# Create a simple test case
client = chromadb.Client()
test_collection = client.create_collection(name="test_semantic")

# Documents with specific technical terms
documents = [
    "The PyTorch framework provides autograd functionality",
    "TensorFlow uses computational graphs for execution",
    "JAX enables functional transformations of NumPy code",
    "PyTorch 2.0 introduced torch.compile for optimization"
]

# Add to collection
test_collection.add(
    documents=documents,
    ids=[f"doc_{i}" for i in range(len(documents))]
)

# Query that should find exact match
query = "PyTorch autograd"

results = test_collection.query(query_texts=[query], n_results=2)
print(f"Query: {query}")
print(f"\nResults:")
for doc, dist in zip(results['documents'][0], results['distances'][0]):
    print(f"- {doc} (distance: {dist:.4f})")

## 2. BM25 Search Implementation

BM25 (Best Matching 25) is a probabilistic retrieval model that:
- Uses term frequency (TF) and inverse document frequency (IDF)
- Excels at exact keyword matching
- Well-understood and fast
- Complementary to semantic search

### How BM25 Works

In [ ]:
from rank_bm25 import BM25Okapi
import re

def tokenize(text):
    """Simple tokenization for BM25."""
    return re.findall(r'\w+', text.lower())

# Prepare corpus for BM25
corpus = [
    "The PyTorch framework provides autograd functionality for automatic differentiation",
    "TensorFlow uses computational graphs for efficient execution and deployment",
    "JAX enables functional transformations of NumPy code for high-performance computing",
    "PyTorch 2.0 introduced torch.compile for graph-based optimization"
]

# Tokenize corpus
tokenized_corpus = [tokenize(doc) for doc in corpus]

# Create BM25 object
bm25 = BM25Okapi(tokenized_corpus)

# Test with query
query = "PyTorch autograd"
tokenized_query = tokenize(query)

# Get BM25 scores
scores = bm25.get_scores(tokenized_query)

print(f"Query: {query}")
print(f"\nBM25 Scores:")
for i, (doc, score) in enumerate(zip(corpus, scores)):
    print(f"{i+1}. Score: {score:.4f} - {doc[:60]}...")

In [ ]:
def bm25_search(corpus, query, top_k=3):
    """BM25 search function."""
    tokenized_corpus = [tokenize(doc) for doc in corpus]
    bm25 = BM25Okapi(tokenized_corpus)
    
    tokenized_query = tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    
    # Get top-k results
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    for idx in top_indices:
        results.append({
            'index': idx,
            'text': corpus[idx],
            'score': float(scores[idx])
        })
    
    return results

# Compare semantic vs BM25
print("=== Semantic Search vs BM25 ===\n")

# Semantic search
query = "PyTorch autograd"
semantic_results = test_collection.query(query_texts=[query], n_results=2)

print("Semantic Search Results:")
for doc, dist in zip(semantic_results['documents'][0], semantic_results['distances'][0]):
    print(f"- {doc[:60]}... (distance: {dist:.4f})")

print("\nBM25 Search Results:")
bm25_results = bm25_search(corpus, query, top_k=2)
for result in bm25_results:
    print(f"- {result['text'][:60]}... (score: {result['score']:.4f})")

## 3. Hybrid Search Architecture

### Why Combine Both?

| Method | Strengths | Weaknesses |
|--------|-----------|------------|
| **Semantic** | Meaning understanding, synonyms | Misses exact terms |
| **BM25** | Exact matches, technical terms | No semantic understanding |
| **Hybrid** | Best of both worlds | More complex |

### Fusion Strategies

In [ ]:
def reciprocal_rank_fusion(results_list1, results_list2, k=60):
    """
    Reciprocal Rank Fusion (RRF) algorithm.
    
    RRF score = Σ 1/(k + rank_i)
    """
    all_docs = {}
    
    # Process first result list
    for rank, result in enumerate(results_list1):
        doc = result['text']
        if doc not in all_docs:
            all_docs[doc] = {'text': doc, 'rrf_score': 0}
        all_docs[doc]['rrf_score'] += 1 / (k + rank + 1)
    
    # Process second result list
    for rank, result in enumerate(results_list2):
        doc = result['text']
        if doc not in all_docs:
            all_docs[doc] = {'text': doc, 'rrf_score': 0}
        all_docs[doc]['rrf_score'] += 1 / (k + rank + 1)
    
    # Sort by RRF score
    sorted_docs = sorted(all_docs.values(), key=lambda x: x['rrf_score'], reverse=True)
    
    return sorted_docs


def weighted_fusion(results_list1, results_list2, weight1=0.7, weight2=0.3):
    """Weighted score fusion with normalization."""
    all_docs = {}
    
    # Normalize and combine scores
    max_score1 = max(r['score'] for r in results_list1) if results_list1 else 1
    max_score2 = max(r['score'] for r in results_list2) if results_list2 else 1
    
    for result in results_list1:
        doc = result['text']
        normalized_score = result['score'] / max_score1
        if doc not in all_docs:
            all_docs[doc] = {'text': doc, 'weighted_score': 0}
        all_docs[doc]['weighted_score'] += weight1 * normalized_score
    
    for result in results_list2:
        doc = result['text']
        normalized_score = result['score'] / max_score2
        if doc not in all_docs:
            all_docs[doc] = {'text': doc, 'weighted_score': 0}
        all_docs[doc]['weighted_score'] += weight2 * normalized_score
    
    sorted_docs = sorted(all_docs.values(), key=lambda x: x['weighted_score'], reverse=True)
    
    return sorted_docs


# Test hybrid search
print("=== Hybrid Search Demonstration ===\n")

# Get results from both methods
semantic_results = []
for doc, dist in zip(semantic_results['documents'][0], semantic_results['distances'][0]):
    semantic_results.append({'text': doc, 'score': 1 - dist})  # Convert distance to similarity

bm25_results = bm25_search(corpus, query, top_k=3)

# Apply fusion
print("Query:", query)
print("\n--- Reciprocal Rank Fusion (RRF) ---")
rrf_results = reciprocal_rank_fusion(semantic_results, bm25_results)
for i, result in enumerate(rrf_results[:3]):
    print(f"{i+1}. {result['text'][:60]}... (RRF: {result['rrf_score']:.4f})")

print("\n--- Weighted Fusion (70% semantic, 30% BM25) ---")
weighted_results = weighted_fusion(semantic_results, bm25_results)
for i, result in enumerate(weighted_results[:3]):
    print(f"{i+1}. {result['text'][:60]}... (Weighted: {result['weighted_score']:.4f})")

## 4. Advanced Chunking Strategies

### Contextual Chunking with Summaries

Instead of just splitting text, we can:
1. Add context from surrounding chunks
2. Generate summaries for each chunk
3. Create parent-child relationships

This improves retrieval accuracy by providing more context.

In [ ]:
def contextual_chunking(text, chunk_size=500, overlap=100):
    """
    Create chunks with surrounding context.
    """
    # First, create base chunks
    from utils.chunking import chunk_text
    base_chunks = chunk_text(text, method="recursive", chunk_size=chunk_size)
    
    contextual_chunks = []
    for i, chunk in enumerate(base_chunks):
        # Add context from previous and next chunks
        context_parts = []
        
        if i > 0:
            context_parts.append(f"Previous context: ...{base_chunks[i-1][-100:]}")
        
        context_parts.append(f"Current content: {chunk}")
        
        if i < len(base_chunks) - 1:
            context_parts.append(f"Next context: {base_chunks[i+1][:100]}...")
        
        contextual_chunks.append({
            'text': chunk,
            'context': "\n".join(context_parts),
            'chunk_index': i
        })
    
    return contextual_chunks

# Demonstrate contextual chunking
sample_text = docs[0]['text']
contextual_chunks = contextual_chunking(sample_text, chunk_size=300)

print(f"Created {len(contextual_chunks)} contextual chunks\n")
for i, chunk in enumerate(contextual_chunks[:2]):
    print(f"--- Chunk {i+1} ---")
    print(f"Main text: {chunk['text'][:100]}...")
    print(f"Context preview: {chunk['context'][:150]}...")
    print()

## 5. Complete Hybrid RAG Pipeline

Now let's build a complete RAG system that combines:
- Semantic search (Gemini embeddings)
- BM25 search
- Hybrid fusion
- Advanced chunking

In [ ]:
class HybridRAG:
    """
    Complete Hybrid RAG system combining semantic and BM25 search.
    """
    
    def __init__(self):
        self.chroma_client = chromadb.Client()
        self.collection = None
        self.corpus = []
        
    def index_documents(self, documents, chunk_method="recursive", chunk_size=300):
        """Index documents with both semantic embeddings and BM25."""
        print("Indexing documents...")
        
        # Chunk documents
        all_chunks = []
        for doc in documents:
            chunks = chunk_text(doc['text'], method=chunk_method, chunk_size=chunk_size)
            all_chunks.extend(chunks)
        
        self.corpus = all_chunks
        
        # Create ChromaDB collection
        self.collection = self.chroma_client.create_collection(name="hybrid_rag")
        
        # Generate embeddings
        print("Generating embeddings...")
        embeddings = []
        for chunk in all_chunks:
            result = genai.embed_content(
                model="models/text-embedding-004",
                content=chunk,
                task_type="retrieval_document"
            )
            embeddings.append(result['embedding'])
        
        # Add to ChromaDB
        self.collection.add(
            documents=all_chunks,
            embeddings=embeddings,
            ids=[f"chunk_{i}" for i in range(len(all_chunks))]
        )
        
        print(f"Indexed {len(all_chunks)} chunks")
    
    def search(self, query, top_k=3, fusion_method="rrf"):
        """Hybrid search combining semantic and BM25."""
        # Semantic search
        query_embedding = genai.embed_content(
            model="models/text-embedding-004",
            content=query,
            task_type="retrieval_query"
        )['embedding']
        
        semantic_results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k
        )
        
        semantic_list = []
        for doc, dist in zip(semantic_results['documents'][0], semantic_results['distances'][0]):
            semantic_list.append({'text': doc, 'score': 1 - dist})
        
        # BM25 search
        bm25_results = bm25_search(self.corpus, query, top_k=top_k)
        
        # Fusion
        if fusion_method == "rrf":
            fused = reciprocal_rank_fusion(semantic_list, bm25_results)
        else:
            fused = weighted_fusion(semantic_list, bm25_results)
        
        return fused[:top_k]
    
    def generate_answer(self, query, search_results):
        """Generate answer using retrieved context."""
        context = "\n\n".join([r['text'] for r in search_results])
        
        prompt = f"""Answer the question based on the provided context.
        
Context:
{context}

Question: {query}

Answer:"""
        
        model = genai.GenerativeModel('gemini-1.5-pro')
        response = model.generate_content(prompt)
        
        return response.text


# Test the hybrid RAG system
hybrid_rag = HybridRAG()
hybrid_rag.index_documents(docs)

# Test queries
test_queries = [
    "What is PyTorch autograd?",
    "How do neural networks work?",
    "What are the types of machine learning?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    
    results = hybrid_rag.search(query, top_k=3)
    answer = hybrid_rag.generate_answer(query, results)
    
    print(f"\nAnswer: {answer[:200]}...")
    print(f"\nRetrieved chunks:")
    for i, result in enumerate(results):
        print(f"  {i+1}. {result['text'][:80]}...")

## 6. Retrieval Evaluation

Let's measure how well our retrieval system performs.

In [ ]:
def precision_at_k(retrieved, relevant, k):
    """Calculate precision@k."""
    retrieved_at_k = retrieved[:k]
    relevant_retrieved = len(set(retrieved_at_k) & set(relevant))
    return relevant_retrieved / k


def recall_at_k(retrieved, relevant, k):
    """Calculate recall@k."""
    retrieved_at_k = retrieved[:k]
    relevant_retrieved = len(set(retrieved_at_k) & set(relevant))
    return relevant_retrieved / len(relevant) if relevant else 0


# Create a simple evaluation dataset
evaluation_data = [
    {
        "query": "What is machine learning?",
        "relevant_keywords": ["machine learning", "AI", "artificial intelligence", "learning"]
    },
    {
        "query": "How do neural networks work?",
        "relevant_keywords": ["neural networks", "layers", "neurons", "deep learning"]
    }
]

print("=== Retrieval Evaluation ===\n")

for eval_item in evaluation_data:
    query = eval_item["query"]
    relevant_keywords = eval_item["relevant_keywords"]
    
    # Get retrieval results
    results = hybrid_rag.search(query, top_k=5)
    retrieved_texts = [r['text'].lower() for r in results]
    
    # Check if relevant keywords appear
    relevant_chunks = []
    for i, text in enumerate(retrieved_texts):
        if any(keyword in text for keyword in relevant_keywords):
            relevant_chunks.append(i)
    
    # Calculate metrics
    retrieved_indices = list(range(len(results)))
    p_at_3 = precision_at_k(retrieved_indices, relevant_chunks, 3)
    r_at_3 = recall_at_k(retrieved_indices, relevant_chunks, 3)
    
    print(f"Query: {query}")
    print(f"  Precision@3: {p_at_3:.3f}")
    print(f"  Recall@3: {r_at_3:.3f}")
    print()

## Summary

In this notebook, we learned:

1. **BM25 Search**: Keyword-based retrieval for exact matches
2. **Hybrid Search**: Combining semantic and BM25 with RRF/weighted fusion
3. **Advanced Chunking**: Contextual chunking with surrounding text
4. **Hybrid RAG Pipeline**: Complete system with both search methods
5. **Evaluation**: Measuring retrieval quality with precision/recall

## Key Takeaways
- Hybrid search often outperforms pure semantic search
- BM25 catches exact matches that embeddings might miss
- RRF is a robust fusion method
- Chunking strategy significantly impacts retrieval quality

## Next Steps
- Move to Session 3 for production monitoring with MLflow!